<a href="https://colab.research.google.com/github/Rinosa123/Bilingual-Enterprise-RAG-Copilot/blob/main/notebooks/04_grounded_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grounded Arabic–English Answer Generation

This notebook evaluates grounded bilingual answer generation using retrieved
enterprise evidence.

## Objectives

1. Load the tested bilingual prompt builder from the project.
2. Load Qwen3-4B-Instruct using 4-bit quantization.
3. Generate English answers for English questions.
4. Generate Arabic answers for Arabic questions.
5. Require inline chunk citations.
6. Test insufficient-evidence refusal behavior.
7. Validate that generated citations exist in the supplied evidence.

The generation experiment is evaluated separately from retrieval so that
generation errors can be distinguished from retrieval errors.

In [1]:
%cd /content

!if [ -d "Bilingual-Enterprise-RAG-Copilot/.git" ]; then \
    git -C Bilingual-Enterprise-RAG-Copilot pull --ff-only origin main; \
else \
    git clone https://github.com/Rinosa123/Bilingual-Enterprise-RAG-Copilot.git; \
fi

%cd /content/Bilingual-Enterprise-RAG-Copilot

!pip -q install \
    "transformers>=4.51.0,<5.0" \
    "accelerate>=1.0.0" \
    "bitsandbytes>=0.46.0"

/content
Cloning into 'Bilingual-Enterprise-RAG-Copilot'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 75 (delta 28), reused 45 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 65.80 KiB | 635.00 KiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/Bilingual-Enterprise-RAG-Copilot
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which

In [2]:
import sys

import bitsandbytes
import torch
import transformers


print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Bitsandbytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is unavailable. Select Runtime → Change runtime type → T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Bitsandbytes: 0.50.0
CUDA available: True
GPU: Tesla T4


In [3]:
from pathlib import Path

from src.generation.prompt_builder import (
    build_grounded_messages,
    get_refusal_response,
)
from src.ingestion.chunker import chunk_documents
from src.ingestion.text_loader import load_text_documents


PROJECT_ROOT = Path.cwd()
DOCUMENT_DIRECTORY = PROJECT_ROOT / "data" / "sample_docs"

documents = load_text_documents(DOCUMENT_DIRECTORY)
chunks = chunk_documents(documents)

chunks_by_id = {
    chunk.chunk_id: chunk
    for chunk in chunks
}

print("Project root:", PROJECT_ROOT)
print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("Available chunk IDs:")

for chunk_id in chunks_by_id:
    print("-", chunk_id)

Project root: /content/Bilingual-Enterprise-RAG-Copilot
Documents: 2
Chunks: 10
Available chunk IDs:
- HR-AR-001-CH-001
- HR-AR-001-CH-002
- HR-AR-001-CH-003
- HR-AR-001-CH-004
- HR-AR-001-CH-005
- HR-EN-001-CH-001
- HR-EN-001-CH-002
- HR-EN-001-CH-003
- HR-EN-001-CH-004
- HR-EN-001-CH-005


In [4]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model.eval()

model_memory_gb = model.get_memory_footprint() / (1024 ** 3)

print("Model loaded successfully.")
print("Model device:", model.device)
print(
    "Loaded in 4-bit:",
    getattr(model, "is_loaded_in_4bit", False),
)
print(f"Model memory footprint: {model_memory_gb:.2f} GB")

Loading: Qwen/Qwen3-4B-Instruct-2507


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded successfully.
Model device: cuda:0
Loaded in 4-bit: True
Model memory footprint: 2.42 GB


In [5]:
def generate_grounded_answer(
    question: str,
    evidence_chunks: list,
    max_new_tokens: int = 160,
) -> str:
    """Generate a deterministic answer grounded in supplied evidence."""

    if not evidence_chunks:
        return get_refusal_response(question)

    messages = build_grounded_messages(
        question=question,
        evidence_chunks=evidence_chunks,
    )

    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    model_inputs = {
        name: tensor.to(model.device)
        for name, tensor in model_inputs.items()
    }

    prompt_length = model_inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    answer_token_ids = generated_ids[
        0,
        prompt_length:,
    ]

    answer = tokenizer.decode(
        answer_token_ids,
        skip_special_tokens=True,
    )

    return answer.strip()


print("Grounded generation function is ready.")

Grounded generation function is ready.


In [6]:
english_question = (
    "How many annual leave days do "
    "full-time employees receive?"
)

english_evidence = [
    chunks_by_id["HR-EN-001-CH-003"],
]

print("Question:")
print(english_question)

print("\nEvidence:")
for chunk in english_evidence:
    print(f"[{chunk.chunk_id}] {chunk.section}")
    print(chunk.text)

english_answer = generate_grounded_answer(
    question=english_question,
    evidence_chunks=english_evidence,
)

print("\nGrounded answer:")
print(english_answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:
How many annual leave days do full-time employees receive?

Evidence:
[HR-EN-001-CH-003] 2. Annual Leave
2. Annual Leave
Full-time employees receive 24 working days of annual leave after completing one year of service. Leave requests should normally be submitted at least 10 working days before the planned leave. All leave requests require approval from the employee's line manager.

Grounded answer:
Full-time employees receive 24 working days of annual leave after completing one year of service [HR-EN-001-CH-003].


In [7]:
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

print("Deterministic generation configuration applied.")

Deterministic generation configuration applied.


In [8]:
arabic_question = (
    "ما الحد الأقصى لتكلفة الفندق لليلة الواحدة؟"
)

arabic_evidence = [
    chunks_by_id["HR-AR-001-CH-003"],
]

print("السؤال:")
print(arabic_question)

print("\nالأدلة:")
for chunk in arabic_evidence:
    print(f"[{chunk.chunk_id}] {chunk.section}")
    print(chunk.text)

arabic_answer = generate_grounded_answer(
    question=arabic_question,
    evidence_chunks=arabic_evidence,
)

print("\nالإجابة المستندة إلى الأدلة:")
print(arabic_answer)

السؤال:
ما الحد الأقصى لتكلفة الفندق لليلة الواحدة؟

الأدلة:
[HR-AR-001-CH-003] 2. السفر في مهام العمل
2. السفر في مهام العمل
يجب الحصول على موافقة المدير قبل حجز أي رحلة عمل. تُستخدم الدرجة الاقتصادية للرحلات التي تقل مدتها عن خمس ساعات. الحد الأقصى لتكلفة الفندق هو 450 ريالاً سعودياً لليلة الواحدة، ما لم تتم الموافقة مسبقاً على مبلغ أعلى.

الإجابة المستندة إلى الأدلة:
الحد الأقصى لتكلفة الفندق لليلة الواحدة هو 450 ريالاً سعودياً، ما لم تتم الموافقة مسبقاً على مبلغ أعلى [HR-AR-001-CH-003].


In [9]:
import re
from typing import Any


CITATION_PATTERN = re.compile(
    r"\[([A-Z]{2}-[A-Z]{2}-\d{3}-CH-\d{3})\]"
)


def extract_citations(answer: str) -> list[str]:
    """Extract unique chunk IDs while preserving their order."""
    return list(
        dict.fromkeys(
            CITATION_PATTERN.findall(answer)
        )
    )


def validate_answer_citations(
    answer: str,
    evidence_chunks: list,
) -> dict[str, Any]:
    """Check whether every citation belongs to supplied evidence."""
    cited_chunk_ids = extract_citations(answer)

    allowed_chunk_ids = {
        chunk.chunk_id
        for chunk in evidence_chunks
    }

    unsupported_citations = [
        chunk_id
        for chunk_id in cited_chunk_ids
        if chunk_id not in allowed_chunk_ids
    ]

    return {
        "citations": cited_chunk_ids,
        "allowed_chunk_ids": sorted(allowed_chunk_ids),
        "unsupported_citations": unsupported_citations,
        "has_citation": bool(cited_chunk_ids),
        "citations_valid": (
            bool(cited_chunk_ids)
            and not unsupported_citations
        ),
    }


english_validation = validate_answer_citations(
    answer=english_answer,
    evidence_chunks=english_evidence,
)

arabic_validation = validate_answer_citations(
    answer=arabic_answer,
    evidence_chunks=arabic_evidence,
)

print("English answer validation:")
print(english_validation)

print("\nArabic answer validation:")
print(arabic_validation)

English answer validation:
{'citations': ['HR-EN-001-CH-003'], 'allowed_chunk_ids': ['HR-EN-001-CH-003'], 'unsupported_citations': [], 'has_citation': True, 'citations_valid': True}

Arabic answer validation:
{'citations': ['HR-AR-001-CH-003'], 'allowed_chunk_ids': ['HR-AR-001-CH-003'], 'unsupported_citations': [], 'has_citation': True, 'citations_valid': True}


In [10]:
unsupported_question = (
    "What is the company's maternity leave policy?"
)

irrelevant_evidence = [
    chunks_by_id["HR-EN-001-CH-003"],
]

expected_refusal = get_refusal_response(
    unsupported_question
)

model_refusal = generate_grounded_answer(
    question=unsupported_question,
    evidence_chunks=irrelevant_evidence,
)

empty_retrieval_refusal = generate_grounded_answer(
    question=unsupported_question,
    evidence_chunks=[],
)

print("Question:")
print(unsupported_question)

print("\nExpected refusal:")
print(expected_refusal)

print("\nModel response with irrelevant evidence:")
print(model_refusal)

print(
    "\nModel followed exact refusal:",
    model_refusal == expected_refusal,
)

print("\nResponse when retrieval returns no chunks:")
print(empty_retrieval_refusal)

print(
    "\nEmpty-retrieval refusal correct:",
    empty_retrieval_refusal == expected_refusal,
)

Question:
What is the company's maternity leave policy?

Expected refusal:
I could not find sufficient evidence in the provided documents to answer this question.

Model response with irrelevant evidence:
I could not find sufficient evidence in the provided documents to answer this question.

Model followed exact refusal: True

Response when retrieval returns no chunks:
I could not find sufficient evidence in the provided documents to answer this question.

Empty-retrieval refusal correct: True


In [11]:
arabic_unsupported_question = (
    "ما هي سياسة إجازة الأمومة في الشركة؟"
)

arabic_irrelevant_evidence = [
    chunks_by_id["HR-AR-001-CH-003"],
]

arabic_expected_refusal = get_refusal_response(
    arabic_unsupported_question
)

arabic_model_refusal = generate_grounded_answer(
    question=arabic_unsupported_question,
    evidence_chunks=arabic_irrelevant_evidence,
)

print("السؤال:")
print(arabic_unsupported_question)

print("\nالرفض المتوقع:")
print(arabic_expected_refusal)

print("\nإجابة النموذج:")
print(arabic_model_refusal)

print(
    "\nالرفض مطابق تماماً:",
    arabic_model_refusal == arabic_expected_refusal,
)

السؤال:
ما هي سياسة إجازة الأمومة في الشركة؟

الرفض المتوقع:
لم أجد أدلة كافية في المستندات المقدمة للإجابة عن هذا السؤال.

إجابة النموذج:
لم أجد أدلة كافية في المستندات المقدمة للإجابة عن هذا السؤال.

الرفض مطابق تماماً: True


In [12]:
import pandas as pd

from src.generation.prompt_builder import (
    detect_question_language,
)


def evaluate_generation_case(
    test_name: str,
    question: str,
    answer: str,
    evidence_chunks: list,
    expected_text: str,
    expects_refusal: bool = False,
) -> dict:
    """Evaluate language, content and citation behaviour."""
    question_language = detect_question_language(question)
    answer_language = detect_question_language(answer)

    same_language = (
        question_language == answer_language
    )

    citations = extract_citations(answer)

    if expects_refusal:
        requirement_passed = (
            answer == get_refusal_response(question)
        )
        citation_policy_passed = not citations
    else:
        requirement_passed = expected_text in answer

        citation_result = validate_answer_citations(
            answer=answer,
            evidence_chunks=evidence_chunks,
        )

        citation_policy_passed = citation_result[
            "citations_valid"
        ]

    overall_passed = all(
        [
            same_language,
            requirement_passed,
            citation_policy_passed,
        ]
    )

    return {
        "Test": test_name,
        "Question Language": question_language,
        "Answer Language": answer_language,
        "Same Language": same_language,
        "Content/Refusal Passed": requirement_passed,
        "Citation Policy Passed": citation_policy_passed,
        "Overall Passed": overall_passed,
    }


evaluation_rows = [
    evaluate_generation_case(
        test_name="English grounded answer",
        question=english_question,
        answer=english_answer,
        evidence_chunks=english_evidence,
        expected_text="24 working days",
    ),
    evaluate_generation_case(
        test_name="Arabic grounded answer",
        question=arabic_question,
        answer=arabic_answer,
        evidence_chunks=arabic_evidence,
        expected_text="450",
    ),
    evaluate_generation_case(
        test_name="English irrelevant-evidence refusal",
        question=unsupported_question,
        answer=model_refusal,
        evidence_chunks=irrelevant_evidence,
        expected_text=expected_refusal,
        expects_refusal=True,
    ),
    evaluate_generation_case(
        test_name="English empty-retrieval refusal",
        question=unsupported_question,
        answer=empty_retrieval_refusal,
        evidence_chunks=[],
        expected_text=expected_refusal,
        expects_refusal=True,
    ),
    evaluate_generation_case(
        test_name="Arabic irrelevant-evidence refusal",
        question=arabic_unsupported_question,
        answer=arabic_model_refusal,
        evidence_chunks=arabic_irrelevant_evidence,
        expected_text=arabic_expected_refusal,
        expects_refusal=True,
    ),
]

evaluation_table = pd.DataFrame(evaluation_rows)

display(evaluation_table)

passed_tests = int(
    evaluation_table["Overall Passed"].sum()
)
total_tests = len(evaluation_table)

print(
    f"\nGeneration checks passed: "
    f"{passed_tests}/{total_tests}"
)

,Test,Question Language,Answer Language,Same Language,Content/Refusal Passed,Citation Policy Passed,Overall Passed
0,English grounded answer,en,en,True,True,True,True
1,Arabic grounded answer,ar,ar,True,True,True,True
2,English irrelevant-evidence refusal,en,en,True,True,True,True
3,English empty-retrieval refusal,en,en,True,True,True,True
4,Arabic irrelevant-evidence refusal,ar,ar,True,True,True,True



Generation checks passed: 5/5


## Results

The grounded generation layer passed all five functional checks:

| Capability | Result |
|---|---:|
| English same-language answer | Passed |
| Arabic same-language answer | Passed |
| Evidence-only citations | Passed |
| Irrelevant-evidence refusal | Passed |
| Empty-retrieval refusal | Passed |

Qwen3-4B-Instruct-2507 was loaded using 4-bit NF4 quantization with an
approximately 2.42 GB model memory footprint on a Tesla T4 GPU.

The English and Arabic grounded answers contained the expected policy
information and cited only chunk IDs supplied in the evidence. When relevant
evidence was unavailable, the system returned the predefined refusal response
without generating unsupported citations.

## Limitations

This is a focused functional evaluation using two synthetic enterprise
documents and five checks. The 5/5 result should not be interpreted as general
production accuracy. A larger bilingual evaluation set, adversarial questions,
retrieval integration, latency measurement and human review are still required.

## Next Step

Connect multilingual retrieval, reranking and grounded generation into one
end-to-end Arabic–English RAG pipeline.